In [ ]:
import numpy as np
import math
from math import sqrt
import cmath
import scipy
import pandas as pd
import csv
import os
from numpy import linalg
from qutip import *
from numpy.linalg import matrix_rank
from collections import Counter
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
import time
import seaborn as sns
from sklearn.metrics import confusion_matrix

# sklearn.preprocessing provides utility functions and transformer classes to change raw feature vectors into a representation that is more suitalb for downstream estimators
# LabelEncoder assigns a real number to the features which are not of number datatype
from sklearn.preprocessing import LabelEncoder   
# Keras Classifier is a wrapper functions which trains the models on subset(s) of training set, basically it tries to find that combination of features which are most important for making predictions                                               
from keras.wrappers.scikit_learn import KerasClassifier 
from keras.utils import np_utils 
#importing the libraries for modelling 
from keras.models import Sequential   # IDK
# Dense means, a neuron in a particular hidden layer is connected to every neuron in the previous layer
# Dropout means, input of some neurons in a particular layer will be set to 0 with some probability/rate  =p 
from keras.layers import Dense, Dropout, BatchNormalization 
#Stop training when a monitered metric such as 'loss' or 'accuracy' has stopped improving. 
from keras.callbacks import EarlyStopping 

from sklearn.metrics import classification_report
from tensorflow import keras



In [ ]:
!pip install qutip

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
#state_vector = (2*np.random.random(2**n)-1) + (2*np.random.random(2**n)-1)*1j 

#state_vector = 
norm = linalg.norm(state_vector)
state_vector /= norm
state_vector = Qobj(state_vector)
rho_ = ket2dm(state_vector)
return rho_.data.toarray()

In [ ]:
n = 1
(2*np.random.random(2**n)-1) + (2*np.random.random(2**n)-1)*1j 

array([-0.70523689+0.44082292j,  0.68340548-0.7930276j ])

In [ ]:
b = 1
a = -1 
x = (b - a) * np.random.random_sample() + a

def rand_coeff(min, max):
  real_coeff = (max - min) * np.random.random_sample() + min
  img_coeff = 1j*((max - min) * np.random.random_sample() + min)
  coeff = real_coeff + img_coeff
  return coeff

In [ ]:
a1 = rand_coeff(min=-1, max=1)
a2 = rand_coeff(min=-1, max=1)
a3 = rand_coeff(min=-1, max=1)
a4 = rand_coeff(min=-1, max=1)
a5 = rand_coeff(min=-1, max=1)

rand_state_0 = np.array([a1, 0 ,0, 0, a2, a3, a4, a5])
rand_state_1 = Qobj(rand_state_0/linalg.norm(rand_state_0))
rho_state_1 =  ket2dm(rand_state_1)
rho_state_2 = rho_state_1.data.toarray()
#rho_state_2

In [ ]:
tol_psd = 1e-10
tol_idem = 1e-10
M = rho_state_2
sh, eig, psd, tr, he, rank, idem, tr  = check_rho(M,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

 Shape: (8, 8),
 PSD: True ,
 trace: (1+2.3787572868201425e-18j) ,
 hermitian: True ,
 eigen values:
 [-6.61779693e-17 -2.08337493e-17 -5.97634129e-18 -2.18975666e-33
  7.77495350e-33  4.07074809e-17  1.07791730e-16  1.00000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (1+2.3787572868201425e-18j)


In [ ]:
three_tangle(Qobj(M))

(0.4646015327537115-2.425250855320596e-17j)

In [ ]:
#M1 = M.reshape([[2,2], [2,2], [2,2]])

#.data.toarray()
#bs1_check(Qobj(M))
#bs1_check(Qobj(M1))

In [ ]:
#M = M.reshape(8,8).shape
M = M.reshape(8,8)

In [ ]:
i = 2
(matrix_rank(Qobj(M).ptrace([i]),hermitian=True))

IndexError: ignored

In [ ]:
M1 = RHO_FS()
#.shape
bs1_check(Qobj(M1.reshape([[2,2],[2,2],[2,2]])))

TypeError: ignored

In [ ]:
def bs1_check(M):
    
    # defining a list to store the ranks 
    rank_list=[]
    
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
        
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[0] == rank_list[1] == 2 and rank_list[2] == 1:
        return 1
    else:
        return 0 

In [ ]:
def Partial_transposition(M): 
    N = np.zeros((4,4), dtype = complex)    
    for i in range(4): 
        for j in range(4): 
            N[i][j] = M[i][j]
    # Putting the partial transpose operation by brute force        
    N[0][1]= M[1][0]               
    N[1][0]= M[0][1] 
    
# defining functions for checking positive-semi-definiteness 
def PSD(M, tol):
    eigen_val = np.linalg.eigvalsh(M)
    if np.all(eigen_val >= -tol ) == True :
        return True 
    else :
        return False 

def check_valid_rho(M, tol_psd):
    
    shape = M.shape
    eigen_values = np.linalg.eigvalsh(M)
    psd = PSD(M, tol_psd)
    trace = M.trace()
    herm = np.all(M == M.conj().T)
    
    return shape, eigen_values, psd, trace, herm

def check_pure_rho(M, tol_idem):
    
    rank = np.linalg.matrix_rank(M)
    M_square = np.matmul(M,M)
    idempotent = np.all(abs(M_square - M) <= tol_idem )
    #idempotent = M_square == M
    trace = M_square.trace()
    
    return rank, idempotent, trace

def check_rho(M,tol_psd, tol_idem):
    
    '''
    print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},
          \n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))
    '''
    sh, eig, psd, tr, he = check_valid_rho(rho_fs, tol_psd)
    rank, idem, tr =  check_pure_rho(rho_fs, tol_idem)
    return sh, eig, psd, tr, he, rank, idem, tr 

def CNOT_23(M):
    CNOT =    np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    CNOT_d =  np.conjugate(CNOT).T
    Identity = np.eye(2)
    CNOT = Qobj(CNOT)
    CNOT_d = Qobj(CNOT)
    Id = Qobj(Identity)
    U = Qobj(tensor(Id, CNOT).data.toarray().reshape(8,8))
    U_d = U.dag()
    Oper = U*M*U_d
    
    return Oper

def CNOT_12(M):
    CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
    #CNOT_d =  np.conjugate(CNOT).T
    Identity = np.eye(2)
    CNOT = Qobj(CNOT)
    Id = Qobj(Identity)
    U = Qobj(tensor(CNOT,Id).data.toarray().reshape(8,8))
    U_d = U.dag()
    Oper = U*M*U_d
    
    return Oper

def CNOT_13(M):
    # taken from https://quantumcomputing.stackexchange.com/questions/4252/how-to-derive-the-cnot-matrix-for-a-3-qubit-system-where-the-control-target-qu
    CNOT_13   = np.array([[1,0,0,0,0,0,0,0],[0,0,0,0,0,1,0,0],[0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,1],
                       [0,0,0,0,1,0,0,0],[0,2,0,0,0,0,0,0],[0,0,0,0,0,0,1,0],[0,0,0,1,0,0,0,0]])
    CNOT_13   = Qobj(CNOT_13)
    Operation = CNOT_13*M*(CNOT_13.dag())
    return Operation

def CNOT_13_state(M):
    CNOT_13   = np.array([[1,0,0,0,0,0,0,0],[0,0,0,0,0,1,0,0],[0,0,1,0,0,0,0,0],[0,0,0,0,0,0,0,1],
                       [0,0,0,0,1,0,0,0],[0,2,0,0,0,0,0,0],[0,0,0,0,0,0,1,0],[0,0,0,1,0,0,0,0]])
    CNOT_13   = Qobj(CNOT_13)
    Operation = CNOT_13*M
    return Operation

def rho(n):
    state_vector = (2*np.random.random(2**n)-1) + (2*np.random.random(2**n)-1)*1j 
    norm = linalg.norm(state_vector)
    state_vector /= norm
    state_vector = Qobj(state_vector)
    rho_ = ket2dm(state_vector)
    return rho_.data.toarray()

def RHO_FS():
    rho_a          = Qobj(rho(1))
    rho_b          = Qobj(rho(1))
    rho_c          = Qobj(rho(1))
    rho_ab         = tensor(rho_a, rho_b)
    rho_abc        = tensor(rho_ab, rho_c)
    return rho_abc.data.toarray()

def rand_bs3():
    rho_bs3 = Qobj(RHO_FS().reshape(8,8), dims = [[8],[8]])
    rho_bs3 = CNOT_12(rho_bs3)
    return rho_bs3.data.toarray()

def rand_bs1():
    rho_bs1 = Qobj(RHO_FS().reshape(8,8), dims = [[8],[8]])
    rho_bs1 = CNOT_23(rho_bs1)
    return rho_bs1.data.toarray()

def swap_12(M):
    swap = Qobj(np.array([[1,0,0,0],[0,0,1,0], [0,1,0,0], [0,0,0,1]]))
    Identity = Qobj(np.eye(2))
    U = Qobj(tensor(swap, Identity).data.toarray(), dims =  [[8], [8]])
    Ud = U.dag()
    oper = U*M*Ud
    return oper

def rand_bs2():
    rho_bs2 = Qobj(RHO_FS().reshape(8,8), dims = [[8],[8]])
    rho_bs2 = CNOT_23(rho_bs2)
    rho_bs2 = swap_12(rho_bs2)
    return rho_bs2.data.toarray()

def w():
    #cnot_12 = CNOT_12(M)
    #Operation = CNOT_13(cnot_12)
    w = w_state(3).data.toarray()
    w[0][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[7][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w = w/np.linalg.norm(w)
    w = ket2dm(Qobj(w, dims = [[2,2,2],[1,1,1]]))
    return w.data.toarray()

def W():
    w = w_state(3).data.toarray()
    w[0][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[1][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[2][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w[4][0] = (2*np.random.random(1)-1) + (2*np.random.random(1)-1)*1j 
    w = w/np.linalg.norm(w)
    w = ket2dm(Qobj(w, dims = [[2,2,2],[1,1,1]]))
    return w.data.toarray()

def array_to_label_list(Matrix,n,k):
    '''
    Matrix refers to the density matrix which is supposed to be input in this function, 
    of whose the list is to be constructed
    
    n = Number of qubits in the system
    
    k = index reference to the three qubit entanglement class 
        k = 0 => FS => Full Separable => A-B-C => [1,0,0,0,0,0]
        k = 1 => bs2 => Bi-Separable-1 => A-BC => [0,1,0,0,0,0]
        k = 2 => BS2 => Bi-Separable-2 => AB-C => [0,0,1,0,0,0]
        k = 3 => BS3 => Bi-Separable-3 => AC-B => [0,0,0,1,0,0]
        k = 4 => W => Werner State => ABC =>      [0,0,0,0,1,0]
        k = 5 => w => Greenberg-Horne-Zeilinger State => ABC => [0,0,0,0,0,1]
        
    From 0 to 31 elements of the returned list real components of the density matrix(Re(rho_00),Re(rho_01),...) i.e, from 0 to N-1
    from 32 to 63 : Imaginary components of the density matrix i.e, (Im(rho_00),Im(rho_01),...) 
    from 64 to 69 : One hot encoded labels corresponding to the class of the 3 qubit entangled states
    '''
    N = 2**(2*n)
    #Matrix   = Matrix.data.toarray()
    Matrix_1 = Matrix.reshape((1,N))
    data = np.zeros((2*N))
    FS  = [1,0,0,0,0,0,0]
    bs2 = [0,1,0,0,0,0,1]
    BS2 = [0,0,1,0,0,0,2]
    BS3 = [0,0,0,1,0,0,3]
    W   = [0,0,0,0,1,0,4]
    w = [0,0,0,0,0,1,5]
    
    for i in range(0,N):
        data[i] = np.real(Matrix_1[0][i])
    for j in range(N,2*N):
        data[j] = np.imag(Matrix_1[0][j-N])
        
    data  = data.tolist()
    if k == 0:
        data.extend(FS)
    elif k == 1:
        data.extend(bs2)
    elif k == 2:
        data.extend(BS2)
    elif k == 3:
        data.extend(BS3)
    elif k == 4:
        data.extend(W)
    elif k == 5:
        data.extend(w)
    else :
        return "Error : Corresponding state label doesn't exist, select from 0,1,2,3,4 or 5"
    return data

def labels(n):
    
    N=2**(2*n)
    fields = [None]*((2*N))
    #list_classes_0 = ['Fully-Separable', 'Bi-Separable-1','Bi-Separable-2','Bi-Separable-3','W','w']
    list_classes_1 = ['FS','BS1','BS2','BS3','W','w','CODE']
     
    k = 0 
    for i in range(2**n):
        for j in range(2**n):
            fields[k] = 'Rho_'+str(i)+'_'+str(j)+'_Real'
            k=k+1
            
    k = N
    for i in range(2**n):
        for j in range(2**n):
            fields[k] = 'Rho_'+str(i)+'_'+str(j)+'_Imag'
            k=k+1
    
    fields.extend(list_classes_1)
    
    return fields

def class_return_3q(k):
    
    if k == 0:
        return RHO_FS()
    elif k == 1:
        return rand_bs1()
    elif k == 2:
        return rand_bs2()
    elif k == 3:
        return rand_bs3()
    elif k == 4:
        return W()
    elif k == 5:
        return w()
    else :
        return "Class index out of [0,1,2,3,4,5]"
    
def data_generator_3q_class(num, k):
    data_list = []
    
    '''
    K = The type of entanglement class one would want to generate with this
            function
    k = 0 => FS => Full Separable => A-B-C => [1,0,0,0,0,0]
    k = 1 => bs2 => Bi-Separable-1 => A-BC => [0,1,0,0,0,0]
    k = 2 => BS2 => Bi-Separable-2 => AB-C => [0,0,1,0,0,0]
    k = 3 => BS3 => Bi-Separable-3 => AC-B => [0,0,0,1,0,0]
    k = 4 => W => W State => ABC =>      [0,0,0,0,1,0]
    k = 5 => w => Greenberg-Horne-Zeilinger State => ABC => [0,0,0,0,0,1]
    Num = No. of quantum states of the class required 
    '''
    
    for i in range(num):
        Matrix = class_return_3q(k)
        data = array_to_label_list(Matrix=Matrix,n=3,k=k)
        data_list.append(data)
    return data_list


# defining functions for checking positive-semi-definiteness 
def PSD(M, tol):
    eigen_val = np.linalg.eigvalsh(M)
    if np.all(eigen_val >= -tol ) == True :
        return True 
    else :
        return False 

def check_valid_rho(M, tol_psd):
    
    shape = M.shape
    eigen_values = np.linalg.eigvalsh(M)
    psd = PSD(M, tol_psd)
    trace = M.trace()
    herm = np.all(M == M.conj().T)
    
    return shape, eigen_values, psd, trace, herm

def check_pure_rho(M, tol_idem):
    
    rank = np.linalg.matrix_rank(M)
    M_square = np.matmul(M,M)
    idempotent = np.all(abs(M_square - M) <= tol_idem )
    #idempotent = M_square == M
    trace = M_square.trace()
    
    return rank, idempotent, trace

def check_rho(M,tol_psd, tol_idem):
    
    '''
    print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},
          \n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))
    '''
    sh, eig, psd, tr, he = check_valid_rho(M, tol_psd)
    rank, idem, tr =  check_pure_rho(M, tol_idem)
    return sh, eig, psd, tr, he, rank, idem, tr 


def df_to_matrix(dataframe):
    
    all_rho_list_real = []
    all_rho_list_imag = []
    loop_length = dataframe.shape[0]
    
    for h in range(0, loop_length):
        rho_list_real = []
        for i in range(0,8):
            for j in range(0,8):
                col_name = 'Rho_'+str(i)+'_'+str(j)+'_Real'
                rho_list_real.append(dataframe.iloc[[h]][col_name].iloc[0])
        all_rho_list_real.append(rho_list_real)
        
    for h in range(0, loop_length):
        rho_list_imag = []
        for i in range(0,8):
            for j in range(0,8):
                col_name = 'Rho_'+str(i)+'_'+str(j)+'_Imag'
                rho_list_imag.append(dataframe.iloc[[h]][col_name].iloc[0])
        all_rho_list_imag.append(rho_list_imag)
        
    return all_rho_list_real, all_rho_list_imag


def tangle_bipartition(rho):
    
    rho = Qobj(rho, dims = [[2,2,2],[2,2,2]])
    rho_reduced = {}
    bi_tangle = {}
    for i in range(3):
        key = 'rho'+str(i)
        rho_reduced[key] = rho.ptrace([i])
    rho_red_list = list(rho_reduced.values())
    bi_tangle['tangle(rho_0_12)'] = 4*np.linalg.det(rho_red_list[0])
    bi_tangle['tangle(rho_1_02)'] = 4*np.linalg.det(rho_red_list[1])
    bi_tangle['tangle(rho_2_01)'] = 4*np.linalg.det(rho_red_list[2])
                                      
    #i_tangle = dict((k, 4*np.linalg.det(v)) for k, v in rho_reduced.items())   

    return bi_tangle

#prhoint('rhoank of rhoho_A:{0}\nrhoank of rhoho_B:{1}\nrhoank of rhoho_C:{2}'.forhomat(rho_A, rho_B, rho_C))

def tangle_pairwise(rho):
    
    '''
     defined for three qubit pure states as given in the article 
     https://arxiv.org/pdf/quant-ph/9907047.pdf
    '''
    rho_red = []
    rho_red_tilda = []
    rho_red_product = []
    rho_red_pro_eig = []
    rho_red_pro_trace = []
    rho_red_tangle = []
    tangle_pair = []
    
    pair_tangle = {}
    j=[1,2,0]
    
    U = tensor(sigmay(),sigmay())
    Ud = U.dag()
    rho = Qobj(rho.data.toarray(), dims = [[2,2,2],[2,2,2]])
    for i in range(3):
        
        rho_red.append(rho.ptrace([i,j[i]])) 
        rho_red_tilda.append(U*(rho_red[i].conj())*Ud)
        rho_red_product.append(rho_red[i]*rho_red_tilda[i])
        evalue, evector = (rho_red_product[i]).eigenstates()
        rho_red_pro_eig.append(evalue)
        #rho_red_pro_trace.append(rho_red_product[i].tr())
        #rho_red_tangle.append(rho_red_pro_trace[i] -2*(rho_red_pro_eig[i][2]*rho_red_pro_eig[i][3]))
        rho_red_pro_eig[i] = sorted(list(rho_red_pro_eig[i]),reverse=True)
        k = rho_red_pro_eig[i]
        tangle_pair.append((max(k[0]-(k[1]+k[2]+k[3]),0))**2)
    
    pair_tangle['tangle(rho_01)'] = tangle_pair[0]
    pair_tangle['tangle(rho_12)'] = tangle_pair[1]
    pair_tangle['tangle(rho_20)'] = tangle_pair[2]
    
    return pair_tangle

def three_tangle(rho):
    three_t = []
    bi_t , pi_t = tangle_bipartition(rho) , tangle_pairwise(rho)
    bi_t_list, pi_t_list = list(bi_t.values()), list(pi_t.values())
    three_t = [bi_t_list[0]-(pi_t_list[0]+pi_t_list[2]), bi_t_list[1]-(pi_t_list[1]+pi_t_list[0]), bi_t_list[2]-(pi_t_list[2]+pi_t_list[1])]   
    three_tangle_avg = np.mean(three_t)
    return three_tangle_avg


# Function to check the full separability of the generated state above 
# based on the metrics in the paper https://arxiv.org/pdf/quant-ph/0005115.pdf

def fs_check(M):
    
    # defining a list to store the ranks 
    rank_list=[]
    
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
        
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[0] == rank_list[1] == rank_list[2] == 1:
        return 1
    
    else:
        return 0 

#return rank_list
#RHO_FS_A = RHO_FS.ptrace([0])

def bs1_check(M):
    
    # defining a list to store the ranks 
    rank_list=[]
    
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
        
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[0] == rank_list[1] == 2 and rank_list[2] == 1:
        return 1
    else:
        return 0 
    
    
def bs2_check(M):
    rank_list=[]
    for i in range(0,3):
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    if rank_list[0] == 1 and rank_list[1] == rank_list[2] == 2:
        return 1
    else:
        return 0 
    
def bs3_check(M):

    rank_list=[]
    for i in range(0,3):
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    if rank_list[1] == 1 and rank_list[0] == rank_list[2] == 2:
        return 1
    else:
        return 0 
    
def table(M):
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    s_a = matrix_rank(M.ptrace([0]))
    s_b = matrix_rank(M.ptrace([1]))
    s_c = matrix_rank(M.ptrace([2]))
    three_t = three_tangle(M)
    list_all = [s_a, s_b, s_c, three_t]
    return s_a, s_b, s_c, three_t


def table_1(M):
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    s_a = matrix_rank(M.ptrace([0]), hermitian = True)
    s_b = matrix_rank(M.ptrace([1]), hermitian = True)
    s_c = matrix_rank(M.ptrace([2]), hermitian = True)
    three_t = three_tangle(M)
    list_all = [s_a, s_b, s_c, three_t]
    return list_all

def det_rho_abc(M):
    det_list=[]
    for i in range(0,3):
        det_list.append(np.real(linalg.det(M.ptrace([i])))) 
        #det_list = np.real(det_list)              
    return det_list[0],det_list[1],det_list[1]

def von_neumann_entropy(rho, base, sparse_condition):
    '''
    expects rho in numpy array form, 
    base = base of the logarithm
    sparse condition is either true or false
    '''
    rho = Qobj(rho)
    vne = entropy_vn(rho, base=base,sparse = sparse_condition)
    return vne

# https://link.springer.com/article/10.1023/A:1025422606845

def wit_w(rho):
    exp = np.matmul(rho.reshape(8,8), ((3/4)*identity(8).data.toarray() - w())).trace()
    return exp

def wit_w(rho):
    exp = np.matmul(rho.reshape(8,8), ((2/3)*identity(8).data.toarray() - W())).trace()
    return exp

def negativity(M):
    eig_list = []
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    eig_0, vec_0 = np.linalg.eig(M.ptrace([0]).data.toarray())
    eig_1, vec_1 = np.linalg.eig(M.ptrace([1]).data.toarray())
    eig_2, vec_2 = np.linalg.eig(M.ptrace([2]).data.toarray())
    eig_list.extend(eig_0)
    eig_list.extend(eig_1)
    eig_list.extend(eig_2)
    min_eig = np.min(eig_list)
    return min_eig